# Variant E ~100M (500k, Unified Tokenizer)
RUN_TRAINING = True
RUN_RESUME_TRAINING = True
- unified custom-delta tokenization (`event_size=4`, `vocab_size=374`),
- structural prefix tokens (Density/Voices/Register + START),
- strict non-fallback dense Variant E profile near 100M params,
- step-level checkpoints for true mid-epoch resume.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

import numpy as np
import torch

REQUIRE_REAL_GDN = True


def _ensure_packages() -> None:
    required = ["pretty_midi", "safetensors", "gradio", "ncps"]
    missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing,
        ])


def _find_root() -> Path:
    probes = [
        Path.cwd().resolve(),
        Path("/kaggle/working/piano_midi_model"),
        Path("/kaggle/input/datasets/chickaboomcmurtrie/magic-midi"),
        Path("/kaggle/input/magic_midi/piano_midi_model"),
        Path("/kaggle/input/magic-midi"),
        Path("/kaggle/input"),
    ]
    anchor = Path("training/train_variant_e_100m_ddp.py")
    for probe in probes:
        if not probe.exists():
            continue
        for p in [probe, *probe.parents]:
            if (p / anchor).exists():
                return p
    raise FileNotFoundError("Project root not found")


def _ensure_flash_linear_attention(require_real_gdn=True) -> None:
    from model.blocks import gdn_block

    if gdn_block.GDN_AVAILABLE:
        print("Real GDN kernels already available.")
        return

    install_specs = ["flash-linear-attention", "flash-linear-attention==0.4.2"]
    last_error = None
    for spec in install_specs:
        try:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                spec,
            ])
            importlib.invalidate_caches()
            if gdn_block.try_import_fla():
                print(f"Installed {spec} and loaded real GDN kernels.")
                return
            print(f"Install succeeded for {spec} but GDN import is still unavailable.")
        except Exception as exc:
            last_error = exc
            print(f"{spec} install failed: {exc}")

    if require_real_gdn:
        raise RuntimeError(
            "flash-linear-attention/GDN backend unavailable in strict mode after install attempts. "
            "Try restarting the kernel and rerunning Cell 2, or switch to fallback mode."
        ) from last_error


def _read_json_file(path: Path):
    return json.loads(path.read_text(encoding="utf-8-sig"))


def _safe_int(value, default=-1):
    try:
        return int(value)
    except Exception:
        return int(default)


def _discover_chunk_ranges(pretokenized_root: Path):
    ranges = []
    seen = set()
    work_roots = [
        pretokenized_root / "pulse88_tokenize_500k_work" / "work",
        pretokenized_root / "work",
    ]
    for work_root in work_roots:
        if not work_root.exists():
            continue
        for chunk_dir in sorted(work_root.glob("chunk_*_*")):
            parts = chunk_dir.name.split("_")
            if len(parts) < 3:
                continue
            start = _safe_int(parts[1], -1)
            end = _safe_int(parts[2], -1)
            if start < 0 or end < start:
                continue
            key = str(chunk_dir.resolve())
            if key in seen:
                continue
            seen.add(key)
            ranges.append((start, end, chunk_dir))
    ranges.sort(key=lambda x: (int(x[0]), int(x[1])))
    return ranges


def _chunk_dir_for_index(index: int, chunk_ranges):
    idx = _safe_int(index, -1)
    if idx < 0:
        return None
    for start, end, chunk_dir in chunk_ranges:
        if int(start) <= idx <= int(end):
            return chunk_dir
    return None


def _resolve_npz_path_for_row(row: dict, *, pretokenized_root: Path, chunk_ranges):
    raw_npz = str(row.get("npz_path", "")).strip()
    md5 = str(row.get("md5", "")).strip()
    idx = _safe_int(row.get("index", -1), -1)
    chunk_dir = _chunk_dir_for_index(idx, chunk_ranges)

    probe_paths = []
    seen = set()

    def _add(path: Path):
        key = str(path)
        if key not in seen:
            seen.add(key)
            probe_paths.append(path)

    if raw_npz:
        candidate = Path(raw_npz)
        if candidate.is_absolute():
            _add(candidate)

        rel_text = raw_npz.lstrip("./\\")
        if rel_text:
            rel = Path(rel_text)
            if chunk_dir is not None:
                _add(chunk_dir / rel)
            _add(pretokenized_root / rel)
            _add(pretokenized_root / "pulse88_tokenize_500k_work" / rel)
            _add(pretokenized_root / "work" / rel)

        raw_norm = raw_npz.replace("\\", "/")
        if "/work/" in raw_norm:
            suffix = raw_norm.split("/work/", 1)[1]
            _add(pretokenized_root / "pulse88_tokenize_500k_work" / "work" / Path(suffix))
            _add(pretokenized_root / "work" / Path(suffix))

        if "/data/" in raw_norm:
            data_suffix = raw_norm.split("/data/", 1)[1]
            if chunk_dir is not None:
                _add(chunk_dir / "data" / Path(data_suffix))
            _add(pretokenized_root / "data" / Path(data_suffix))

    if md5:
        if chunk_dir is not None:
            _add(chunk_dir / f"{md5}.npz")
            _add(chunk_dir / "data" / f"{md5}.npz")
        _add(pretokenized_root / f"{md5}.npz")
        _add(pretokenized_root / "data" / f"{md5}.npz")

    for path in probe_paths:
        if path.exists() and path.is_file():
            return path.resolve()
    return None


def _fast_guess_npz_path_for_row(row: dict, *, pretokenized_root: Path, chunk_ranges):
    raw_npz = str(row.get("npz_path", "")).strip()
    idx = _safe_int(row.get("index", -1), -1)
    chunk_dir = _chunk_dir_for_index(idx, chunk_ranges)

    if raw_npz:
        candidate = Path(raw_npz)
        if candidate.is_absolute():
            return candidate.resolve()

        rel_text = raw_npz.lstrip("./\\")
        if rel_text and chunk_dir is not None:
            return (chunk_dir / Path(rel_text)).resolve()

        raw_norm = raw_npz.replace("\\", "/")
        if raw_norm.startswith("pulse88_tokenize_500k_work/work/"):
            suffix = raw_norm.split("pulse88_tokenize_500k_work/work/", 1)[1]
            return (pretokenized_root / "pulse88_tokenize_500k_work" / "work" / Path(suffix)).resolve()
        if raw_norm.startswith("work/"):
            suffix = raw_norm.split("work/", 1)[1]
            return (pretokenized_root / "work" / Path(suffix)).resolve()

    return None


def _normalize_manifest_rows(rows, *, pretokenized_root: Path):
    chunk_ranges = _discover_chunk_ranges(pretokenized_root)
    normalized = []
    unresolved = 0
    missing_npz = 0
    fast_guess_used = 0
    fast_verify_budget = 512
    fast_verified = 0

    for row in rows:
        if not isinstance(row, dict):
            unresolved += 1
            continue

        resolved_npz = _fast_guess_npz_path_for_row(
            row,
            pretokenized_root=pretokenized_root,
            chunk_ranges=chunk_ranges,
        )
        if resolved_npz is not None:
            fast_guess_used += 1
            if fast_verify_budget > 0:
                fast_verified += 1
                fast_verify_budget -= 1
                if not resolved_npz.exists() or not resolved_npz.is_file():
                    resolved_npz = _resolve_npz_path_for_row(
                        row,
                        pretokenized_root=pretokenized_root,
                        chunk_ranges=chunk_ranges,
                    )
        else:
            resolved_npz = _resolve_npz_path_for_row(
                row,
                pretokenized_root=pretokenized_root,
                chunk_ranges=chunk_ranges,
            )
        if resolved_npz is None:
            raw_npz = str(row.get("npz_path", "")).strip()
            if not raw_npz and not str(row.get("md5", "")).strip():
                missing_npz += 1
            else:
                unresolved += 1
            continue

        clean = dict(row)
        clean["npz_path"] = str(resolved_npz)
        if "tokens" not in clean and "length" in clean:
            clean["tokens"] = clean["length"]
        normalized.append(clean)

    print(
        f"Manifest normalization: kept={len(normalized)} unresolved={unresolved} missing_npz={missing_npz} fast_guess={fast_guess_used} verified={fast_verified}"
    )
    return normalized


def _manifest_sample_has_resolution_keys(rows, sample_limit: int = 64):
    for row in rows[:max(1, int(sample_limit))]:
        if not isinstance(row, dict):
            continue
        if str(row.get("npz_path", "")).strip() or str(row.get("md5", "")).strip():
            return True
    return False


def _manifest_has_resolved_rows(rows, *, min_found: int = 2, sample_limit: int = 2048):
    found = 0
    checked = 0
    for row in rows:
        if not isinstance(row, dict):
            continue
        raw_npz = str(row.get("npz_path", "")).strip()
        if not raw_npz:
            continue
        p = Path(raw_npz)
        if p.is_file():
            found += 1
            if found >= int(max(1, min_found)):
                return True
        checked += 1
        if checked >= int(max(1, sample_limit)):
            break
    return False


def _load_manifest_rows(*, primary_manifest: Path, pretokenized_root: Path, fallback_manifest: Path):
    if fallback_manifest.exists():
        try:
            cached_payload = _read_json_file(fallback_manifest)
            if isinstance(cached_payload, list) and cached_payload and _manifest_has_resolved_rows(cached_payload):
                print(f"Using cached normalized manifest: {fallback_manifest} rows={len(cached_payload)}")
                return cached_payload, fallback_manifest
            print("Cached fallback manifest exists but appears unresolved; rebuilding.")
        except Exception as exc:
            print(f"Cached fallback manifest load failed ({fallback_manifest}): {exc}")

    if primary_manifest.exists():
        payload = _read_json_file(primary_manifest)
        if isinstance(payload, list) and payload:
            if _manifest_sample_has_resolution_keys(payload):
                if _manifest_has_resolved_rows(payload):
                    print(f"Using primary manifest directly: {primary_manifest} rows={len(payload)}")
                    return payload, primary_manifest
                payload = _normalize_manifest_rows(payload, pretokenized_root=pretokenized_root)
                if payload:
                    fallback_manifest.parent.mkdir(parents=True, exist_ok=True)
                    fallback_manifest.write_text(json.dumps(payload), encoding="utf-8")
                    return payload, fallback_manifest
            else:
                print(f"Primary manifest {primary_manifest} looks controller-style; skipping direct normalization.")

    merged_rows = []
    chunk_manifest_paths = []
    seen_paths = set()

    chunk_bases = [
        pretokenized_root / "pulse88_tokenize_500k_work" / "work",
        pretokenized_root / "work",
        pretokenized_root / "tokenized",
        pretokenized_root,
    ]

    for base in chunk_bases:
        if not base.exists():
            continue
        for manifest_path in sorted(base.glob("chunk_*/metadata/manifest.json")):
            key = str(manifest_path.resolve())
            if key in seen_paths:
                continue
            seen_paths.add(key)
            chunk_manifest_paths.append(manifest_path)

    for manifest_path in chunk_manifest_paths:
        try:
            payload = _read_json_file(manifest_path)
        except Exception:
            continue
        if not isinstance(payload, list):
            continue
        for row in payload:
            if isinstance(row, dict):
                clean = dict(row)
                raw_npz = str(clean.get("npz_path", "")).strip()
                if raw_npz and not Path(raw_npz).is_absolute():
                    clean["npz_path"] = str((manifest_path.parent.parent / Path(raw_npz)).resolve())
                merged_rows.append(clean)

    if merged_rows:
        merged_rows = _normalize_manifest_rows(merged_rows, pretokenized_root=pretokenized_root)
        if merged_rows:
            fallback_manifest.parent.mkdir(parents=True, exist_ok=True)
            fallback_manifest.write_text(json.dumps(merged_rows), encoding="utf-8")
            return merged_rows, fallback_manifest

    raise RuntimeError(
        "No usable pretokenized manifest rows found. Checked controller manifest and chunk manifests. "
        f"Controller manifest path: {primary_manifest}"
    )


_ensure_packages()
PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

_ensure_flash_linear_attention(require_real_gdn=REQUIRE_REAL_GDN)

from model.blocks import gdn_block

if REQUIRE_REAL_GDN and not gdn_block.GDN_AVAILABLE:
    raise RuntimeError("flash-linear-attention/GDN backend unavailable in strict mode")

MAX_PIECES = 500_000
SEED = 42
EPOCHS = 16
BATCH_PER_GPU = 2
TARGET_EFFECTIVE_BATCH = 64
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.015
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.05
SEED_LENGTH = 512
CONTINUATION_LENGTH = 1536
MAX_SEQUENCE_LENGTH = SEED_LENGTH + CONTINUATION_LENGTH
VOCAB_SIZE = 374
EVENT_SIZE = 4
TOKENIZATION_STRATEGY = "custom_delta"
SAVE_EVERY_N_STEPS = 500

RUN_TRAINING = False
RUN_RESUME_TRAINING = True

OUTPUT_DIR = Path("/kaggle/working") / "outputs" / "variant_e_100m_500k_barmeta"
MANIFEST_FALLBACK = OUTPUT_DIR / "source_pretokenized" / "manifest.json"

PRETOKENIZED_ROOT = Path("/kaggle/input/datasets/chickaboomcmurtrie/pulse88-tokenized-500k")
CONTROLLER_ROOT = PRETOKENIZED_ROOT / "pulse88_tokenize_500k_work" / "_controller"
CONTROLLER_MANIFEST = CONTROLLER_ROOT / "manifest_500k.json"
CONTROLLER_SOURCE_INDEX = CONTROLLER_ROOT / "source_index.json"
CONTROLLER_STATE = CONTROLLER_ROOT / "state.json"

if not PRETOKENIZED_ROOT.exists():
    raise FileNotFoundError(f"Expected dataset path not found: {PRETOKENIZED_ROOT}")
if not CONTROLLER_ROOT.exists():
    print(f"Controller directory not found: {CONTROLLER_ROOT}; trying chunk manifests.")

if RUN_TRAINING:
    rows, PRETOKENIZED_MANIFEST = _load_manifest_rows(
        primary_manifest=CONTROLLER_MANIFEST,
        pretokenized_root=PRETOKENIZED_ROOT,
        fallback_manifest=MANIFEST_FALLBACK,
    )
    if not isinstance(rows, list) or not rows:
        raise RuntimeError(f"Manifest is empty or invalid: {PRETOKENIZED_MANIFEST}")
else:
    PRETOKENIZED_MANIFEST = MANIFEST_FALLBACK if MANIFEST_FALLBACK.exists() else CONTROLLER_MANIFEST
    rows = []
    print(f"Resume-only fast path: skipping manifest rebuild; using {PRETOKENIZED_MANIFEST}")

VARIANT_E_100M_PROFILE = {
    "d_model": 1024,
    "n_layers": 14,
    "attention_every_n_layers": 2,
    "gdn_inner_ratio": 0.50,
    "gdn_num_heads": 4,
    "gqa_num_heads": 8,
    "gqa_groups": 4,
    "target_params": 100_000_000,
}

SELECTED = {
    "params": 100_000_000,
    "profile": VARIANT_E_100M_PROFILE,
    "backend": {"gdn_using_fallback": False, "cfc_using_fallback": False},
}

gpu_count = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
grad_accum = max(1, int(np.ceil(TARGET_EFFECTIVE_BATCH / max(1, BATCH_PER_GPU * max(1, gpu_count)))))
steps_per_epoch = max(1, int(np.ceil((MAX_PIECES * 0.9) / max(1, BATCH_PER_GPU * max(1, gpu_count)) / grad_accum)))
total_steps = max(1, int(steps_per_epoch * EPOCHS))
warmup_steps = max(1, int(round(WARMUP_RATIO * total_steps)))

RUN_SUMMARY = {
    "PROJECT_ROOT": str(PROJECT_ROOT),
    "PRETOKENIZED_ROOT": str(PRETOKENIZED_ROOT),
    "PRETOKENIZED_MANIFEST": str(PRETOKENIZED_MANIFEST),
    "CONTROLLER_ROOT": str(CONTROLLER_ROOT),
    "CONTROLLER_MANIFEST": str(CONTROLLER_MANIFEST),
    "CONTROLLER_SOURCE_INDEX": str(CONTROLLER_SOURCE_INDEX),
    "CONTROLLER_STATE": str(CONTROLLER_STATE),
    "SELECTED_PROFILE": SELECTED,
    "GPU_COUNT": gpu_count,
    "GRAD_ACCUM": grad_accum,
    "STEPS_PER_EPOCH": steps_per_epoch,
    "TOTAL_STEPS": total_steps,
    "WARMUP_STEPS": warmup_steps,
}
print(json.dumps(RUN_SUMMARY, indent=2))

In [ ]:
# Explicit training cell (set RUN_TRAINING=True to execute)
import importlib
import json
import os
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path

import torch

from hf_sync import normalize_hf_repo_id

if importlib.util.find_spec("miditok") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "miditok",
    ])

RUN_TRAINING = True  # keep enabled so this cell launches training
RUN_RESUME_TRAINING = True

if importlib.util.find_spec("miditok") is None:
    raise RuntimeError("miditok failed to import after installation.")

_ddp_script = PROJECT_ROOT / "training" / "train_variant_e_100m_ddp.py"
if not _ddp_script.exists():
    raise FileNotFoundError(_ddp_script)


def _bootstrap_hf_sync() -> tuple[str, bool, int]:
    hf_sync_repo_id = normalize_hf_repo_id(
        str(os.environ.get("HF_SYNC_REPO_ID", "")).strip()
        or "Chickaboo/Pulse88-E-85M-Alpha"
    )
    if not hf_sync_repo_id:
        raise ValueError("Set HF_SYNC_REPO_ID to a valid Hugging Face model repo id.")

    hf_token = (
        str(os.environ.get("HF_TOKEN", "")).strip()
        or str(os.environ.get("HUGGINGFACE_HUB_TOKEN", "")).strip()
    )
    if not hf_token:
        try:
            hf_token = getpass("Enter Hugging Face token for private checkpoint uploads: ").strip()
        except Exception:
            hf_token = ""

    if hf_token:
        os.environ["HF_TOKEN"] = hf_token

    os.environ["HF_SYNC_REPO_ID"] = hf_sync_repo_id
    hf_sync_private = str(os.environ.get("HF_SYNC_PRIVATE", "1")).strip().lower() in {
        "1",
        "true",
        "yes",
        "on",
    }
    os.environ["HF_SYNC_PRIVATE"] = "1" if hf_sync_private else "0"

    hf_sync_steps_raw = str(os.environ.get("HF_SYNC_EVERY_N_STEPS", "")).strip()
    try:
        hf_sync_steps = int(hf_sync_steps_raw) if hf_sync_steps_raw else int(SAVE_EVERY_N_STEPS)
    except Exception:
        hf_sync_steps = int(SAVE_EVERY_N_STEPS)
    if hf_sync_steps <= 0:
        hf_sync_steps = int(SAVE_EVERY_N_STEPS)
    os.environ["HF_SYNC_EVERY_N_STEPS"] = str(int(hf_sync_steps))
    return hf_sync_repo_id, hf_sync_private, hf_sync_steps


def _resume_priority(path: Path) -> tuple[int, float]:
    bundle_root = path.parent
    score = 0
    if (bundle_root / "checkpoint_sync_summary.json").exists():
        score += 100_000
    if path.name == "latest_state.pt":
        score += 100
    if (bundle_root / "latest.safetensors").exists():
        score += 50
    if (bundle_root / "best.safetensors").exists():
        score += 10
    try:
        mtime = float(path.stat().st_mtime)
    except Exception:
        mtime = 0.0
    return score, mtime


def _find_latest_resume_checkpoint() -> str:
    candidates = []
    for root in [OUTPUT_DIR, Path("/kaggle/input"), PROJECT_ROOT]:
        if root.exists():
            candidates.extend(root.rglob("latest_state.pt"))
    if not candidates:
        return ""
    candidates.sort(key=_resume_priority, reverse=True)
    return str(candidates[0].resolve())


resume_ckpt = _find_latest_resume_checkpoint() if RUN_RESUME_TRAINING else ""

if RUN_TRAINING:
    profile = SELECTED["profile"]
    train_rows, resolved_manifest = _load_manifest_rows(
        primary_manifest=CONTROLLER_MANIFEST,
        pretokenized_root=PRETOKENIZED_ROOT,
        fallback_manifest=MANIFEST_FALLBACK,
    )
    if not isinstance(train_rows, list) or not train_rows:
        raise RuntimeError(f"Manifest is empty or invalid: {resolved_manifest}")
    PRETOKENIZED_MANIFEST = resolved_manifest

    hf_sync_repo_id, hf_sync_private, hf_sync_steps = _bootstrap_hf_sync()
    print(
        f"HF checkpoint mirror target: {hf_sync_repo_id} "
        "(it will be created automatically on the first successful upload if missing)."
    )
    print(f"Using training manifest: {PRETOKENIZED_MANIFEST}")

    cmd = [
        str(_ddp_script),
        "--pretokenized_manifest", str(PRETOKENIZED_MANIFEST),
        "--pretokenized_root", str(PRETOKENIZED_ROOT),
        "--output_dir", str(OUTPUT_DIR),
        "--max_pieces", str(int(MAX_PIECES)),
        "--seed", str(int(SEED)),
        "--seed_length", str(int(SEED_LENGTH)),
        "--continuation_length", str(int(CONTINUATION_LENGTH)),
        "--max_sequence_length", str(int(MAX_SEQUENCE_LENGTH)),
        "--tokenization_strategy", str(TOKENIZATION_STRATEGY),
        "--event_size", str(int(EVENT_SIZE)),
        "--vocab_size", str(int(VOCAB_SIZE)),
        "--d_model", str(int(profile["d_model"])),
        "--n_layers", str(int(profile["n_layers"])),
        "--attention_every_n_layers", str(int(profile["attention_every_n_layers"])),
        "--gdn_inner_ratio", str(float(profile["gdn_inner_ratio"])),
        "--gdn_num_heads", str(int(profile["gdn_num_heads"])),
        "--gqa_num_heads", str(int(profile["gqa_num_heads"])),
        "--gqa_groups", str(int(profile["gqa_groups"])),
        "--full_attention",
        "--epochs", str(int(EPOCHS)),
        "--batch_size", str(int(BATCH_PER_GPU)),
        "--grad_accumulation_steps", str(int(grad_accum)),
        "--learning_rate", str(float(LEARNING_RATE)),
        "--warmup_ratio", str(float(WARMUP_RATIO)),
        "--warmup_steps", str(int(warmup_steps)),
        "--weight_decay", str(float(WEIGHT_DECAY)),
        "--label_smoothing", str(float(LABEL_SMOOTHING)),
        "--save_every_n_steps", str(int(SAVE_EVERY_N_STEPS)),
        "--save_every_n_epochs", "1",
        "--hf_sync_repo_id", hf_sync_repo_id,
        "--hf_sync_every_n_steps", str(int(hf_sync_steps)),
    ]
    if hf_sync_private:
        cmd.append("--hf_sync_private")

    if resume_ckpt:
        cmd.extend(["--resume_from_checkpoint", resume_ckpt, "--resume_mode", "remaining"])
    else:
        cmd.append("--no_auto_resume")

    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        torchrun = shutil.which("torchrun")
        if torchrun:
            full_cmd = [torchrun, "--standalone", "--nnodes=1", "--nproc_per_node", str(torch.cuda.device_count()), *cmd]
        else:
            full_cmd = [
                sys.executable,
                "-m",
                "torch.distributed.run",
                "--standalone",
                "--nnodes=1",
                "--nproc_per_node",
                str(torch.cuda.device_count()),
                *cmd,
            ]
    else:
        full_cmd = [sys.executable, *cmd]

    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run(full_cmd, check=True, cwd=str(PROJECT_ROOT), env=env)
else:
    print("Set RUN_TRAINING=True in Cell 2 to launch training.")
    if resume_ckpt:
        print(f"Latest resume checkpoint candidate: {resume_ckpt}")

result_json = OUTPUT_DIR / "variant_e_100m_ddp_result.json"
if result_json.exists():
    payload = json.loads(result_json.read_text(encoding="utf-8"))
    print(f"Result JSON: {result_json}")
    result = payload.get("result", {})
    losses = result.get("val_loss", []) if isinstance(result, dict) else []
    if isinstance(losses, list) and losses:
        print(f"Best val loss: {min(float(v) for v in losses):.6f}")
else:
    print(f"No result yet. Enable RUN_TRAINING in Cell 2 to launch: {result_json}")

In [ ]:
# Hugging Face checkpoint mirror bootstrap
import os
from getpass import getpass

from hf_sync import normalize_hf_repo_id

HF_SYNC_REPO_ID = normalize_hf_repo_id(
    os.environ.get("HF_SYNC_REPO_ID", "Chickaboo/Pulse88-E-85M-Alpha")
)
if not HF_SYNC_REPO_ID:
    raise ValueError("Set HF_SYNC_REPO_ID to a valid Hugging Face model repo id.")

HF_TOKEN = (
    os.environ.get("HF_TOKEN", "").strip()
    or os.environ.get("HUGGINGFACE_HUB_TOKEN", "").strip()
)
if not HF_TOKEN:
    try:
        HF_TOKEN = getpass("Enter Hugging Face token for private checkpoint uploads: ").strip()
    except Exception:
        HF_TOKEN = ""

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_SYNC_REPO_ID"] = HF_SYNC_REPO_ID
os.environ["HF_SYNC_PRIVATE"] = os.environ.get("HF_SYNC_PRIVATE", "1")

print(f"HF checkpoint mirror repo: {HF_SYNC_REPO_ID}")
print(f"HF token provided: {'yes' if HF_TOKEN else 'no'}")

In [ ]:
# Explicit resume cell (set RUN_RESUME_NOW=True to execute)
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

if importlib.util.find_spec("miditok") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "miditok",
    ])

RUN_RESUME_NOW = True
RESUME_MODE = "remaining"
RESUME_FROM_CHECKPOINT = ""
PREFERRED_INPUT_MODEL_NAME = "Variant-E-100m-Checkpoint"
USE_INPUT_MANIFEST_WHEN_AVAILABLE = True
RESUME_MANIFEST_CACHE = OUTPUT_DIR / "source_pretokenized" / "manifest.json"

_ddp_script = PROJECT_ROOT / "training" / "train_variant_e_100m_ddp.py"
if not _ddp_script.exists():
    raise FileNotFoundError(_ddp_script)


def _normalize_name(text: str) -> str:
    return "".join(ch.lower() for ch in str(text) if ch.isalnum())



def _discover_manifest(bundle_root: Path):
    manifest_candidates = [
        bundle_root / "source_pretokenized" / "manifest.json",
        bundle_root / "processed_pretokenized" / "manifest.json",
        bundle_root / "manifest.json",
        bundle_root.parent / "manifest.json",
        bundle_root.parent / "processed_pretokenized" / "manifest.json",
        bundle_root.parent / "source_pretokenized" / "manifest.json",
    ]
    for path in manifest_candidates:
        if path.exists() and path.is_file():
            return path.resolve()
    return None



def _discover_tokenizer_json(bundle_root: Path):
    candidates = [
        bundle_root / "custom_tokenizer.json",
        bundle_root.parent / "custom_tokenizer.json",
    ]
    for path in candidates:
        if path.exists() and path.is_file():
            return path.resolve()
    return None



def _find_resume_checkpoint_and_assets(preferred_name: str = ""):
    roots = [
        Path("/kaggle/input/models/chickaboomcmurtrie/variant-e-100m-checkpoint/pytorch/default/1"),
        Path("/kaggle/input"),
        OUTPUT_DIR,
        PROJECT_ROOT,
    ]
    state_candidates = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("*state*.pt"):
            key = str(path.resolve())
            if key in seen:
                continue
            seen.add(key)
            state_candidates.append(path.resolve())

    if not state_candidates:
        return None, None, None

    preferred_norm = _normalize_name(preferred_name)
    ranked = []
    for state_path in state_candidates:
        bundle_root = state_path.parent
        manifest_path = _discover_manifest(bundle_root)
        tokenizer_path = _discover_tokenizer_json(bundle_root)

        score = 0
        candidate_norm = _normalize_name(str(state_path))
        if preferred_norm and preferred_norm in candidate_norm:
            score += 1000
        if (bundle_root / "checkpoint_sync_summary.json").exists():
            score += 100000
        if state_path.name == "latest_state.pt":
            score += 100
        if (bundle_root / "latest.safetensors").exists():
            score += 50
        if (bundle_root / "best.safetensors").exists():
            score += 10
        if manifest_path is not None:
            score += 20
        if tokenizer_path is not None:
            score += 10
        try:
            score += int(state_path.stat().st_mtime)
        except Exception:
            pass

        ranked.append((score, state_path, manifest_path, tokenizer_path))

    ranked.sort(key=lambda item: item[0], reverse=True)
    _, best_state, best_manifest, best_tokenizer = ranked[0]
    return best_state, best_manifest, best_tokenizer



def _try_read_global_step(state_path: Path) -> int:
    try:
        payload = torch.load(str(state_path), map_location="cpu")
        return int(payload.get("global_step", -1))
    except Exception:
        return -1



def _ensure_resume_manifest_cache() -> Path:
    if RESUME_MANIFEST_CACHE.exists():
        return RESUME_MANIFEST_CACHE

    rows, resolved_manifest = _load_manifest_rows(
        primary_manifest=CONTROLLER_MANIFEST,
        pretokenized_root=PRETOKENIZED_ROOT,
        fallback_manifest=RESUME_MANIFEST_CACHE,
    )
    if not isinstance(rows, list) or not rows:
        raise RuntimeError(f"Failed to build resume manifest cache: {resolved_manifest}")
    return resolved_manifest



def _resolve_resume_manifest(discovered_manifest: Path | None):
    if discovered_manifest is not None and bool(USE_INPUT_MANIFEST_WHEN_AVAILABLE):
        try:
            payload = _read_json_file(Path(discovered_manifest))
            if isinstance(payload, list) and payload and _manifest_has_resolved_rows(payload):
                print(f"Using input manifest directly: {discovered_manifest} rows={len(payload)}")
                return Path(discovered_manifest)
            print("Input manifest is not pre-resolved; using cached dataset manifest for speed.")
        except Exception as exc:
            print(f"Input manifest read failed; using cached dataset manifest: {exc}")

    return _ensure_resume_manifest_cache()


if RUN_RESUME_NOW:
    profile = SELECTED["profile"]
    explicit_ckpt = str(RESUME_FROM_CHECKPOINT).strip()

    if explicit_ckpt:
        resume_ckpt = Path(explicit_ckpt).expanduser().resolve()
        if not resume_ckpt.exists():
            raise FileNotFoundError(f"Configured RESUME_FROM_CHECKPOINT not found: {resume_ckpt}")
        resume_manifest = _resolve_resume_manifest(None)
        detected_tokenizer = None
    else:
        discovered_ckpt, discovered_manifest, detected_tokenizer = _find_resume_checkpoint_and_assets(
            preferred_name=PREFERRED_INPUT_MODEL_NAME
        )
        if discovered_ckpt is None:
            raise FileNotFoundError(
                "No checkpoint state file found under /kaggle/input, OUTPUT_DIR, or PROJECT_ROOT."
            )
        resume_ckpt = discovered_ckpt
        resume_manifest = _resolve_resume_manifest(discovered_manifest)

    resume_step = _try_read_global_step(resume_ckpt)
    print(f"Resuming from checkpoint: {resume_ckpt}")
    print(f"Resume global_step from state: {resume_step}")
    print(f"Resume manifest: {resume_manifest}")
    if detected_tokenizer is not None:
        print(f"Detected tokenizer json: {detected_tokenizer}")

    cmd = [
        str(_ddp_script),
        "--pretokenized_manifest", str(resume_manifest),
        "--pretokenized_root", str(PRETOKENIZED_ROOT),
        "--output_dir", str(OUTPUT_DIR),
        "--max_pieces", str(int(MAX_PIECES)),
        "--seed", str(int(SEED)),
        "--seed_length", str(int(SEED_LENGTH)),
        "--continuation_length", str(int(CONTINUATION_LENGTH)),
        "--max_sequence_length", str(int(MAX_SEQUENCE_LENGTH)),
        "--tokenization_strategy", str(TOKENIZATION_STRATEGY),
        "--event_size", str(int(EVENT_SIZE)),
        "--vocab_size", str(int(VOCAB_SIZE)),
        "--d_model", str(int(profile["d_model"])),
        "--n_layers", str(int(profile["n_layers"])),
        "--attention_every_n_layers", str(int(profile["attention_every_n_layers"])),
        "--gdn_inner_ratio", str(float(profile["gdn_inner_ratio"])),
        "--gdn_num_heads", str(int(profile["gdn_num_heads"])),
        "--gqa_num_heads", str(int(profile["gqa_num_heads"])),
        "--gqa_groups", str(int(profile["gqa_groups"])),
        "--full_attention",
        "--epochs", str(int(EPOCHS)),
        "--batch_size", str(int(BATCH_PER_GPU)),
        "--grad_accumulation_steps", str(int(grad_accum)),
        "--learning_rate", str(float(LEARNING_RATE)),
        "--warmup_ratio", str(float(WARMUP_RATIO)),
        "--warmup_steps", str(int(warmup_steps)),
        "--weight_decay", str(float(WEIGHT_DECAY)),
        "--label_smoothing", str(float(LABEL_SMOOTHING)),
        "--save_every_n_steps", str(int(SAVE_EVERY_N_STEPS)),
        "--save_every_n_epochs", "1",
        "--resume_from_checkpoint", str(resume_ckpt),
        "--resume_mode", str(RESUME_MODE),
    ]

    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        torchrun = shutil.which("torchrun")
        if torchrun:
            full_cmd = [torchrun, "--standalone", "--nnodes=1", "--nproc_per_node", str(torch.cuda.device_count()), *cmd]
        else:
            full_cmd = [
                sys.executable,
                "-m",
                "torch.distributed.run",
                "--standalone",
                "--nnodes=1",
                "--nproc_per_node",
                str(torch.cuda.device_count()),
                *cmd,
            ]
    else:
        full_cmd = [sys.executable, *cmd]

    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run(full_cmd, check=True, cwd=str(PROJECT_ROOT), env=env)
else:
    print("Set RUN_RESUME_NOW=True to resume from checkpoint input.")

In [ ]:
# Export-to-root copy cell disabled by request.
# Gradio downloader (Cell 6) is preferred for artifact retrieval.
# If needed later, this export block can be restored from notebook history.

print("Export copy cell is disabled. Use the Gradio downloader cell below.")

In [ ]:
# Optional Gradio download UI (useful when Kaggle file panel is slow)
from pathlib import Path
import importlib
import subprocess
import sys

if importlib.util.find_spec("gradio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "gradio"])
gr = importlib.import_module("gradio")

DOWNLOAD_ROOT = Path("/kaggle/working")
ALLOWED_SUFFIXES = {".pt", ".safetensors", ".json", ".mid", ".txt", ".zip", ".gz"}

def _list_downloadable_files(limit: int = 1000):
    files = []
    for p in DOWNLOAD_ROOT.rglob("*"):
        if not p.is_file():
            continue
        suffix = p.suffix.lower()
        if suffix in ALLOWED_SUFFIXES or p.name.endswith(".tar.gz"):
            files.append(p)
    files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return [str(p) for p in files[:limit]]

def _refresh_files():
    files = _list_downloadable_files()
    default = files[0] if files else None
    status = f"Indexed {len(files)} files under {DOWNLOAD_ROOT}"
    return gr.update(choices=files, value=default), status, default

def _select_file(path: str):
    if not path:
        return None
    p = Path(path)
    if not p.exists() or not p.is_file():
        return None
    return str(p)

with gr.Blocks(title="Variant E 100M Artifact Downloader") as demo:
    gr.Markdown("## Variant E 100M Artifact Downloader")
    gr.Markdown("Refresh, pick a file, then use the download widget.")
    refresh_btn = gr.Button("Refresh File List")
    file_dropdown = gr.Dropdown(label="Artifact", choices=[], value=None)
    status_md = gr.Markdown()
    download_file = gr.File(label="Download Selected File")

    refresh_btn.click(_refresh_files, outputs=[file_dropdown, status_md, download_file])
    file_dropdown.change(_select_file, inputs=file_dropdown, outputs=download_file)
    demo.load(_refresh_files, outputs=[file_dropdown, status_md, download_file])

demo.launch(inline=True, share=True, quiet=True)